# Explainable Deep Ensemble Learning for Multi-Scale Streamflow Bias Correction
## GloFAS-ERA5 across Snow-Influenced Transboundary Basins of Central Asia

Reproducible Google Colab walkthrough for the `sbc` framework.

GloFAS-ERA5 is globally skilful yet systematically **underestimates** discharge in small,
snow- and glacier-fed mountain catchments. This notebook corrects that bias across the
**decadal** gauge network of the Syr Darya / Chu / Talas system (primary) and the **daily**
Naryn headwaters (case study), and explains the correction with SHAP attribution tied to
snow processes.

**Design:** log-residual target `log(q_obs) - log(q_glofas)`; baseline ladder (quantile
mapping, scaling) + gradient-boosting trio + EA-LSTM + the flagship regime-aware
probabilistic physically-constrained network + a leakage-safe stacked ensemble; validated
under temporal holdout, leave-one-basin-out and prediction-in-ungauged-regions (PUR).

_Grant BR24993128 (Science Committee, MES RK). Second paper of a Central-Asian water-resources series (after Water 2025, 17, 2097)._

## 1 · Environment

In [ ]:
# On Colab: set SBC_ROOT to the cloned repo (e.g. mounted from Google Drive).
import os, sys
SBC_ROOT = os.environ.get('SBC_ROOT', '/content/MDPI-Q1-2026')
os.environ['SBC_ROOT'] = SBC_ROOT
sys.path.insert(0, os.path.join(SBC_ROOT, 'src'))
# !pip -q install -r {SBC_ROOT}/requirements.txt   # uncomment on a fresh Colab runtime
import sbc; print('sbc', sbc.__version__, '| root:', SBC_ROOT)

## 2 · Copernicus credentials
Create a free ECMWF account; one Personal Access Token works for both portals. Accept the
licences for `cems-glofas-historical` (EWDS) and `reanalysis-era5-land-monthly-means` (CDS).

In [ ]:
from pathlib import Path
secrets = Path(SBC_ROOT) / 'configs' / 'secrets.env'
secrets.write_text(
    'CDS_URL=https://cds.climate.copernicus.eu/api\n'
    'CDS_KEY=YOUR_TOKEN\n'
    'EWDS_URL=https://ewds.climate.copernicus.eu/api\n'
    'EWDS_KEY=YOUR_TOKEN\n')
print('wrote', secrets)

## 3 · Data acquisition
Ground truth (CA-discharge, 24 MB) is on Zenodo; the predictor (GloFAS-ERA5 v4.0) and the
ERA5-Land monthly forcing stream from Copernicus (resumable). The GloFAS pull takes ~2 h.

In [ ]:
import urllib.request
from sbc.config import PATHS
PATHS.ca_discharge.mkdir(parents=True, exist_ok=True)
gpkg = PATHS.ca_discharge / 'CA-discharge.gpkg'
if not gpkg.exists():
    urllib.request.urlretrieve(
        'https://zenodo.org/records/8147591/files/CA-discharge.gpkg?download=1', gpkg)
print('CA-discharge:', gpkg.stat().st_size // 1_000_000, 'MB')

In [ ]:
from sbc.data.glofas import download_glofas, download_uparea
from sbc.data.era5_land import download_era5_land
download_glofas()          # GloFAS-ERA5 v4.0 daily discharge, one NetCDF per year
download_uparea()          # upstream-area map for gauge->river-pixel snapping
download_era5_land()       # ERA5-Land monthly forcing (SWE, snowmelt, T, precip, ...)

## 4 · Build analysis-ready tables
Extract the observed series + static attributes from the GeoPackage, snap each gauge to its
GloFAS river pixel, and aggregate ERA5-Land to the contributing basins.

In [ ]:
from sbc.data import ca_discharge, glofas, era5_land
ca_discharge.main()        # processed/{gauges,discharge_*,static_attributes,basins}.parquet
glofas.main_extract()      # interim/glofas_daily_at_gauges.parquet
era5_land.main_extract()   # interim/era5land_monthly_at_basins.parquet

## 5 · Assemble the modelling table

In [ ]:
from sbc.data.assemble import assemble
decadal = assemble('decadal')
decadal[['code','basin','domain','date','q_obs','q_glofas','log_residual']].head()

## 6 · Controlled synthetic benchmark
A physically-grounded synthetic experiment with a *known* injected GloFAS-style bias verifies,
in a controlled setting, that the framework recovers the bias before touching the real data.

In [ ]:
from sbc.experiment import run_experiment
syn = run_experiment(source='synthetic', scale='decadal', quick=True,
                     models=['scaling','qmap','lgbm'])
syn['summary']

## 7 · Real-data experiment (decadal, primary)
The full model fleet across the temporal / leave-one-basin-out / PUR validation matrix.
Drop `quick=True` and raise the Optuna trial count for the publication run.

In [ ]:
real = run_experiment(source='real', scale='decadal', quick=True,
                      models=['scaling','qmap','lgbm','catboost',
                              'ealstm','regimeprobnet','stacked'])
real['summary']

## 8 · Results — skill before vs after correction

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
res = real['results']
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, split in zip(ax, ['temporal', 'pur']):
    sub = res[res.split == split]
    order = sub.groupby('model')['kge'].median().sort_values().index
    data = [sub[sub.model == m]['kge'].dropna() for m in order]
    a.boxplot(data, labels=order, vert=False)
    a.axvline(sub['kge_raw'].median(), color='r', ls='--', label='raw GloFAS')
    a.set_title(f'KGE\u2032 ({split})'); a.legend(); a.set_xlim(-0.5, 1)
plt.tight_layout(); plt.show()

## 9 · Explainability — SHAP attribution of the snow drivers
Global importance, snow-feature dependence and regime-conditional attribution show that the
correction loads physically onto snow-water-equivalent, snowmelt and temperature in the melt
and rain-on-snow regimes.

In [ ]:
from sbc.explain import shap_analysis as X
from sbc.models.boosting import LightGBMCorrector
from sbc.validation.splits import temporal_split
from sbc.features.engineering import build_features
from sbc.features.regimes import classify_regimes
df = classify_regimes(build_features(decadal, scale='decadal')).reset_index(drop=True)
tr, te = temporal_split(df)
model = LightGBMCorrector().fit(df[tr])
sh = X.tree_shap(model, df[te])
X.global_importance(sh).head(15)

In [ ]:
X.save_beeswarm(sh)
from sbc.config import PATHS
from IPython.display import Image
Image(str(PATHS.shap_dir / 'beeswarm.png'))

## 10 · Notes
* Decadal is the operational irrigation-planning unit in Central Asia and offers spatial
  generalisation (60 gauges / 7 basins); the 2 daily Naryn gauges form a high-resolution case
  study (`scale='daily'`).
* PUR (train on the core Syr Darya/Chu/Talas, test on the held-out Amu Darya domain) is the
  strongest generalisation evidence for this sparse network.
* SHAP shows *associations within the model*, corroborated by the EA-LSTM snow-store dynamics;
  it is reported as physical consistency, not proven causation.